# Limpieza y transformaciones realizadas

Analisis de calidad sobre los CSV en `data/raw/`. Objetivo: detectar problemas y definir que hara el pipeline Silver (flags, validaciones, que conservar).


## Configuracion

In [66]:
import json
import re
from pathlib import Path
from decimal import Decimal

import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", "{:.2f}".format)

EMAIL_RE = re.compile(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")

PK_MAP = {
    "university/semesters": "semester_id",
    "university/professors": "professor_id",
    "university/students": "student_id",
    "university/courses": "course_id",
    "university/enrollments": "enrollment_id",
    "university/grades": "grade_id",
    "billing/customers": "customer_id",
    "billing/products": "product_id",
    "billing/subscriptions": "subscription_id",
    "billing/invoices": "invoice_id",
    "billing/invoice_items": "invoice_item_id",
    "billing/payments": "payment_id",
    "crm/accounts": "account_id",
    "crm/contacts": "contact_id",
    "crm/leads": "lead_id",
    "crm/opportunities": "opportunity_id",
    "crm/activities": "activity_id",
}

PROJECT_ROOT = Path("/home/jovyan/work")
RAW_PATH = PROJECT_ROOT / "data" / "raw"

with open(PROJECT_ROOT / "manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)

DOMAINS = ["university", "billing", "crm"]


def load_raw(domain, table):
    path = RAW_PATH / domain / f"{table}.csv"
    return pd.read_csv(path)


def to_decimal(x):
    return Decimal(str(x))


def list_tables():
    result = []
    for domain in DOMAINS:
        for table in manifest["domains"][domain]:
            result.append((domain, table))
    return result


tables = list_tables()
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Tablas a analizar:", len(tables))

PROJECT_ROOT: /home/jovyan/work
Tablas a analizar: 18


## 1. Valores nulos

In [67]:
null_rows = []

for domain, table in tables:
    df = load_raw(domain, table)
    for col in df.columns:
        n = df[col].isna().sum()
        if n > 0:
            null_rows.append({
                "tabla": f"{domain}/{table}",
                "columna": col,
                "nulos": n,
                "pct": round(100 * n / len(df), 2),
            })

null_df = pd.DataFrame(null_rows).sort_values("pct", ascending=False)
null_df

,tabla,columna,nulos,pct
0,billing/customers,external_ref,5000,50.00
2,crm/activities,opportunity_id,9985,49.92
1,crm/activities,contact_id,5976,29.88


## 2. Duplicados

**LLaves primarias duplicadas**

In [87]:
dup_pk = []

for tabla_key, pk in PK_MAP.items():
    domain, table = tabla_key.split("/")
    df = load_raw(domain, table)
    dup_pk.append({
        "tabla": tabla_key,
        "pk": pk,
        "duplicados": df[pk].duplicated().sum(),
    })

dup_pk_df = pd.DataFrame(dup_pk)
dup_pk_df

,tabla,pk,duplicados
0,university/semesters,semester_id,0
1,university/professors,professor_id,0
2,university/students,student_id,0
3,university/courses,course_id,0
4,university/enrollments,enrollment_id,0
5,university/grades,grade_id,0
6,billing/customers,customer_id,0
7,billing/products,product_id,0
8,billing/subscriptions,subscription_id,0
9,billing/invoices,invoice_id,0


**Emails invalidos y duplicados**

In [69]:
tablas_con_email = [
    ("university", "professors"),
    ("university", "students"),
    ("billing", "customers"),
    ("crm", "contacts"),
    ("crm", "leads"),
]

email_check = []

for domain, table in tablas_con_email:
    df = load_raw(domain, table)
    emails = df["email"]
    vacios = (emails.isna() | (emails.astype(str).str.strip() == "")).sum()
    invalidos = 0
    for mail in emails.dropna():
        if not EMAIL_RE.match(str(mail).strip()):
            invalidos += 1
    email_check.append({
        "tabla": f"{domain}/{table}",
        "vacios": vacios,
        "invalidos": invalidos,
        "duplicados": emails.duplicated().sum(),
    })

email_check_df = pd.DataFrame(email_check)
print("Emails invalidos :", email_check_df["invalidos"].sum())
email_check_df

Emails invalidos : 0


,tabla,vacios,invalidos,duplicados
0,university/professors,0,0,0
1,university/students,0,0,0
2,billing/customers,0,0,0
3,crm/contacts,0,0,2
4,crm/leads,0,0,0


In [70]:
contacts = load_raw("crm", "contacts")
dup_emails = contacts[contacts["email"].duplicated(keep=False)].sort_values("email")
print("Registros con email duplicado en contacts:", len(dup_emails))
dup_emails[["contact_id", "email", "account_id"]]

Registros con email duplicado en contacts: 4


,contact_id,email,account_id
2466,CON-0002467,ignacio.tapia5946@mail.test,ACC-0002715
8391,CON-0008392,ignacio.tapia5946@mail.test,ACC-0004626
3103,CON-0003104,macarena.ortiz4996@lake.local,ACC-0003066
4934,CON-0004935,macarena.ortiz4996@lake.local,ACC-0001325


In [71]:
dup_filas = []

for domain, table in tables:
    df = load_raw(domain, table)
    dup_filas.append({
        "tabla": f"{domain}/{table}",
        "filas_duplicadas": df.duplicated().sum(),
    })

dup_filas_df = pd.DataFrame(dup_filas)
dup_filas_df[dup_filas_df["filas_duplicadas"] > 0]

,tabla,filas_duplicadas


## 3. Formatos, fechas y catalogos

**Verificamos espacios extra en emails**

In [72]:
for domain, table in tablas_con_email:
    df = load_raw(domain, table)
    mal = (df["email"].astype(str) != df["email"].astype(str).str.strip()).sum()
    print(f"{domain}/{table}.email con espacios extra: {mal}")

university/professors.email con espacios extra: 0
university/students.email con espacios extra: 0
billing/customers.email con espacios extra: 0
crm/contacts.email con espacios extra: 0
crm/leads.email con espacios extra: 0


**Verificamos si las fechas son parseables**

In [73]:
date_checks = []

for domain, table in tables:
    df = load_raw(domain, table)
    for col in df.columns:
        if col.endswith("_date") or col.endswith("_at"):
            parsed = pd.to_datetime(df[col], errors="coerce")
            nat = parsed.isna().sum() - df[col].isna().sum()
            if nat > 0:
                date_checks.append({
                    "tabla": f"{domain}/{table}",
                    "columna": col,
                    "nat": nat,
                })

if date_checks:
    date_check_df = pd.DataFrame(date_checks)
    date_check_df
else:
    print("Todas las fechas no nulas se parsean bien con pd.to_datetime")

Todas las fechas no nulas se parsean bien con pd.to_datetime


**Verificamos que las fechas cumplan una secuencia temporal**

In [74]:
subscriptions = load_raw("billing", "subscriptions")
invoices = load_raw("billing", "invoices")

subs = subscriptions.copy()
subs["start_date"] = pd.to_datetime(subs["start_date"], errors="coerce")
subs["end_date"] = pd.to_datetime(subs["end_date"], errors="coerce")
bad_sub_dates = subs[subs["end_date"] < subs["start_date"]]

inv = invoices.copy()
inv["issued_at"] = pd.to_datetime(inv["issued_at"], errors="coerce")
inv["due_at"] = pd.to_datetime(inv["due_at"], errors="coerce")
bad_inv_dates = inv[inv["due_at"] < inv["issued_at"]]

print("subscriptions con end_date < start_date:", len(bad_sub_dates))
print("invoices con due_at < issued_at:", len(bad_inv_dates))

subscriptions con end_date < start_date: 783
invoices con due_at < issued_at: 0


**DESCUBRIMIENTO**  
No existe coherencia temporal en 783 filas en la tabla `subscriptions`

## 4. Integridad referencial (FKs)

**Conteo de llaves foraneas huerfanas**

In [76]:
def count_orphans(child, child_col, parent, parent_col):
    mask = child[child_col].notna()
    orphans = child[mask & ~child[child_col].isin(parent[parent_col])]
    return len(orphans)


students = load_raw("university", "students")
courses = load_raw("university", "courses")
semesters = load_raw("university", "semesters")
professors = load_raw("university", "professors")
enrollments = load_raw("university", "enrollments")
grades = load_raw("university", "grades")
customers = load_raw("billing", "customers")
products = load_raw("billing", "products")
subscriptions = load_raw("billing", "subscriptions")
invoices = load_raw("billing", "invoices")
invoice_items = load_raw("billing", "invoice_items")
payments = load_raw("billing", "payments")
accounts = load_raw("crm", "accounts")
contacts = load_raw("crm", "contacts")
opportunities = load_raw("crm", "opportunities")
opp_contacts = load_raw("crm", "opportunity_contacts")
activities = load_raw("crm", "activities")

fk_checks = [
    ("enrollments", enrollments, "student_id", students, "student_id"),
    ("enrollments", enrollments, "course_id", courses, "course_id"),
    ("enrollments", enrollments, "semester_id", semesters, "semester_id"),
    ("grades", grades, "enrollment_id", enrollments, "enrollment_id"),
    ("courses", courses, "professor_id", professors, "professor_id"),
    ("invoices", invoices, "customer_id", customers, "customer_id"),
    ("invoice_items", invoice_items, "invoice_id", invoices, "invoice_id"),
    ("invoice_items", invoice_items, "product_id", products, "product_id"),
    ("payments", payments, "invoice_id", invoices, "invoice_id"),
    ("subscriptions", subscriptions, "customer_id", customers, "customer_id"),
    ("subscriptions", subscriptions, "product_id", products, "product_id"),
    ("contacts", contacts, "account_id", accounts, "account_id"),
    ("opportunities", opportunities, "account_id", accounts, "account_id"),
    ("opp_contacts", opp_contacts, "opportunity_id", opportunities, "opportunity_id"),
    ("opp_contacts", opp_contacts, "contact_id", contacts, "contact_id"),
]

fk_rows = []
for label, child, ccol, parent, pcol in fk_checks:
    n = count_orphans(child, ccol, parent, pcol)
    fk_rows.append({"relacion": f"{label}.{ccol} -> {pcol}", "huerfanos": n})

fk_df = pd.DataFrame(fk_rows)
fk_df

,relacion,huerfanos
0,enrollments.student_id -> student_id,0
1,enrollments.course_id -> course_id,0
2,enrollments.semester_id -> semester_id,0
3,grades.enrollment_id -> enrollment_id,0
4,courses.professor_id -> professor_id,0
5,invoices.customer_id -> customer_id,0
6,invoice_items.invoice_id -> invoice_id,0
7,invoice_items.product_id -> product_id,0
8,payments.invoice_id -> invoice_id,0
9,subscriptions.customer_id -> customer_id,0


In [77]:
ext_valid = customers.loc[customers["external_ref"].notna(), "external_ref"]
orph_ext = ext_valid[~ext_valid.isin(students["student_id"])].count()
print(f"external_ref huerfanos (no en students): {orph_ext}")
print(f"external_ref validos: {len(ext_valid) - orph_ext}/{len(ext_valid)}")

print("activities.contact_id huerfanos:", count_orphans(activities, "contact_id", contacts, "contact_id"))
print("activities.opportunity_id huerfanos:", count_orphans(activities, "opportunity_id", opportunities, "opportunity_id"))

external_ref huerfanos (no en students): 0
external_ref validos: 5000/5000
activities.contact_id huerfanos: 0
activities.opportunity_id huerfanos: 0


## 5. Outliers (exploratorio)

In [79]:
leads = load_raw("crm", "leads")
accounts = load_raw("crm", "accounts")

outlier_cols = [
    ("grades.score", grades["score"]),
    ("leads.score", leads["score"]),
    ("invoices.total", invoices["total"]),
    ("payments.amount", payments["amount"]),
    ("opportunities.amount", opportunities["amount"]),
    ("accounts.annual_revenue", accounts["annual_revenue"]),
]

outlier_rows = []
for label, serie in outlier_cols:
    s = serie.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    fuera = s[(s < low) | (s > high)]
    outlier_rows.append({
        "columna": label,
        "fuera_iqr": len(fuera),
        "pct": round(100 * len(fuera) / len(s), 2),
    })

outlier_df = pd.DataFrame(outlier_rows)
outlier_df

,columna,fuera_iqr,pct
0,grades.score,217,0.36
1,leads.score,0,0.00
2,invoices.total,3532,7.06
3,payments.amount,6025,7.53
4,opportunities.amount,220,7.33
5,accounts.annual_revenue,461,9.22


**DESCUBRIMIENTO**  
Se encontro varios outliers con la tecnica del rango intercuartilico, pero al tratarse de valores monetarios y calificaciones, aun se debera verificar si son valores correctos segun las reglas de negocio.

## 6. Reglas de negocio

In [80]:
bad_scores = grades[(grades["score"] < 0) | (grades["score"] > 100)]
bad_leads = leads[(leads["score"] < 0) | (leads["score"] > 100)]
neg_invoices = invoices[invoices["total"] < 0]
neg_payments = payments[payments["amount"] < 0]
neg_items = invoice_items[invoice_items["line_total"] < 0]
bad_qty = invoice_items[invoice_items["quantity"] <= 0]

ii = invoice_items.copy()

ii["quantity_dec"] = ii["quantity"].apply(to_decimal)
ii["unit_price_dec"] = ii["unit_price"].apply(to_decimal)
ii["line_total_dec"] = ii["line_total"].apply(to_decimal)

ii["calc_total"] = ii["quantity_dec"] * ii["unit_price_dec"]

ii_bad = ii[ii["line_total_dec"] != ii["calc_total"]]

business_checks = pd.DataFrame([
    {"regla": "grades.score entre 0-100", "violaciones": len(bad_scores)},
    {"regla": "leads.score entre 0-100", "violaciones": len(bad_leads)},
    {"regla": "invoices.total >= 0", "violaciones": len(neg_invoices)},
    {"regla": "payments.amount >= 0", "violaciones": len(neg_payments)},
    {"regla": "invoice_items.line_total >= 0", "violaciones": len(neg_items)},
    {"regla": "invoice_items.quantity > 0", "violaciones": len(bad_qty)},
    {"regla": "line_total = quantity * unit_price", "violaciones": len(ii_bad)},
])
business_checks

,regla,violaciones
0,grades.score entre 0-100,0
1,leads.score entre 0-100,0
2,invoices.total >= 0,0
3,payments.amount >= 0,0
4,invoice_items.line_total >= 0,0
5,invoice_items.quantity > 0,0
6,line_total = quantity * unit_price,10668


**DESCUBRIMIENTO**  
- Al existir 0 violacion en valores de rango para grades.score, se determina que no existen outliers en esta columna.
- Existen 10668 errores de calculo o presicion en line_total

## 7. Reconciliacion financiera

In [88]:
items_sum = (
    invoice_items.groupby("invoice_id")["line_total"]
    .apply(lambda s: sum(to_decimal(x) for x in s))
    .reset_index(name="items_sum_dec")
)
inv_recon = invoices.merge(items_sum, on="invoice_id", how="left")
inv_recon["items_sum_dec"] = inv_recon["items_sum_dec"].apply(
    lambda x: Decimal("0") if pd.isna(x) else x
)
inv_recon["total_dec"] = inv_recon["total"].apply(to_decimal)

exact_match = (inv_recon["total_dec"] == inv_recon["items_sum_dec"]).sum()
print(f"Facturas con total igual a la suma de sus items (Decimal): {exact_match}/{len(inv_recon)}")

sin_items = invoices[~invoices["invoice_id"].isin(invoice_items["invoice_id"])]
print(f"Facturas sin items: {len(sin_items)}")

Facturas con total igual a la suma de sus items (Decimal): 1/50000
Facturas sin items: 2502


In [89]:
pay_sum = (
    payments.groupby("invoice_id")["amount"]
    .apply(lambda s: sum(to_decimal(x) for x in s))
    .reset_index(name="paid_sum_dec")
)
pay_recon = invoices.merge(pay_sum, on="invoice_id", how="left")
pay_recon["paid_sum_dec"] = pay_recon["paid_sum_dec"].apply(
    lambda x: Decimal("0") if pd.isna(x) else x
)
pay_recon["total_dec"] = pay_recon["total"].apply(to_decimal)

overpaid = pay_recon[pay_recon["paid_sum_dec"] > pay_recon["total_dec"]]
underpaid_paid = pay_recon[
    (pay_recon["status"] == "paid") & (pay_recon["paid_sum_dec"] < pay_recon["total_dec"])
]

print(f"Facturas sobrepagadas: {len(overpaid)} ({100*len(overpaid)/len(pay_recon):.1f}%)")
print(f"status=paid pero pago < total: {len(underpaid_paid)} ({100*len(underpaid_paid)/len(pay_recon):.1f}%)")

Facturas sobrepagadas: 20483 (41.0%)
status=paid pero pago < total: 14481 (29.0%)


## 8. Pesos de calificaciones

In [91]:
weight_sum = grades.groupby("enrollment_id")["weight"].apply(
    lambda s: sum(to_decimal(x) for x in s)
)
bad_weights = weight_sum[weight_sum != Decimal("1")]

print(f"Enrollments con suma de weights != 1: {len(bad_weights)}/{len(weight_sum)}")
print(f"({100*len(bad_weights)/len(weight_sum):.1f}%)")

Enrollments con suma de weights != 1: 22646/22786
(99.4%)
